# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/colkeaz/MLInternship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** Ranking Signal Analysis. This notebook writes the contract for one slice of the warehouse
release, verifies it with exactly three queries on the mid-panel month `2026-03`, builds five
features, and then leaks the label once on purpose to see what leakage does to a score.

**How to run.** In Colab, store a plain **Read** Hugging Face token as a Secret named `HF_TOKEN`.
Locally, export `HF_TOKEN` (or put it in the repo's git-ignored `.env`). The token is never typed
into a cell and never printed. The two month files and the small content dimension are fetched once with `huggingface_hub`, then
all SQL runs locally in DuckDB — no 79M-row download and no repeated scans of the hosted release.

Source: `FlyRank/internship-warehouse`, build `flyrank_pseudonymized_warehouse_release_v20260703`.

In [1]:
import os, getpass, warnings
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from huggingface_hub import hf_hub_download

# Token order: env var -> Colab Secret -> prompt. Never written into a cell.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

REPO = "FlyRank/internship-warehouse"
FILES = {
    "mar": hf_hub_download(REPO, "fact_content_daily_performance/month=2026-03/data_0.parquet",
                           repo_type="dataset", token=HF_TOKEN),
    "apr": hf_hub_download(REPO, "fact_content_daily_performance/month=2026-04/data_0.parquet",
                           repo_type="dataset", token=HF_TOKEN),
    "dim": hf_hub_download(REPO, "dim_content.parquet",
                           repo_type="dataset", token=HF_TOKEN),
}

con = duckdb.connect()
for name, path in FILES.items():
    view = {"mar": "month_mar", "apr": "month_apr", "dim": "dim_content"}[name]
    con.execute(f'CREATE OR REPLACE VIEW {view} AS SELECT * FROM read_parquet(\'{path}\')')

schema = con.sql("DESCRIBE month_mar").df()[["column_name", "column_type"]]
display(schema)

required = {"report_date", "client_hash_id", "content_hash_id", "gsc_impressions",
            "gsc_clicks", "gsc_avg_position", "gsc_data_available", "ga4_data_available"}
missing = required - set(schema["column_name"])
assert not missing, f"Unexpected warehouse schema, missing: {sorted(missing)}"
assert {"content_hash_id", "content_created_date", "content_updated_date"} <= set(
    con.sql("DESCRIBE dim_content").df()["column_name"])
print("Setup done. Schema is mechanical here; the three checks start in section 3.")

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


Setup done. Schema is mechanical here; the three checks start in section 3.


## 1. Unit of analysis + time window

The contract in five plain answers, written before the numbers below were read:

**1) What one row means.** One row is **one pseudonymized content item (a page)** that had at least
100 GSC impressions in March 2026 and at least 100 in April 2026. It is not a page-day, not a
query, not a client. March is collapsed into five features; April is collapsed into one outcome.

**2) Which tables.** `fact_content_daily_performance` for the two months — `month=2026-03` gives the
features, `month=2026-04` gives the label — plus `dim_content` for the immutable creation date. The
daily grain is `report_date × client_hash_id × content_hash_id`; my row is that grain summed to one
page per month.

**3) Which time window.** Decision moment **2026-04-01**. Features cover **2026-03-01 → 2026-03-31**;
the label covers **2026-04-01 → 2026-04-30**. The windows do not overlap. **June 2026 is the sealed
final month** and is never read in this notebook.

**4) What I predict or rank.** The label is **impression-weighted `gsc_avg_position` over April**
(lower is better) — a proxy for search positioning, not for content quality or business value.
Pages are ranked by predicted April position. The honest score is MAE on a 25% client-grouped
holdout against a training-median baseline, with R² reported beside it.

**5) What I deliberately exclude.** `fact_content_query_90d`. Its window is fixed at the last ~90 days
of the snapshot, which reaches into the April label and the sealed June month, so its context
columns are measured after the decision moment. Also excluded: the whole GA4 family (proved
impractical in section 3.3), identifier hashes as features, and every June row.

## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `avg_position_mar`, `log_impressions_mar`, `ctr_mar`, `days_with_impressions_mar`, `age_days_at_decision` | Each is a March measurement (or immutable metadata), so each exists before 2026-04-01. |
| **Label / proxy** | impression-weighted `gsc_avg_position`, April 1–30 | The thing being predicted. Never a feature, in any form. |
| **Context** | `client_hash_id`, `content_hash_id` | Joins, grain checks and client-grouped splits only. Never model inputs. |
| **Excluded** | `fact_content_query_90d` columns | Fixed 90-day window overlaps the label month and June. |
| **Excluded** | `ga4_*` columns and `ga4_data_available` | Only 4.2% of March rows are `IS TRUE` (3.3), and zeros before `ga4_data_start` are not "no engagement". |
| **Excluded** | `dim_content` descriptive fields (`word_count`, `content_type`, …) | The table is a 2026-07-03 snapshot; I cannot prove their value on 2026-04-01. |
| **Excluded** | raw IDs as features, any June row | No predictive meaning / sealed test month. |

## 3. Verify it with queries

Exactly three checks, each one query, all on `month=2026-03`. Everything else in this notebook is
either setup or a feature build, and is labelled as such.

### 3.1 Grain — is one row really one page-day, and does it collapse to one page?

Claim: the daily fact is unique at `report_date × client_hash_id × content_hash_id`, and no content
item in March appears under more than one client — so summing March to one row per page is safe.

In [2]:
grain = con.sql(
    """
    WITH duplicate_grain AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM month_mar
        GROUP BY ALL
        HAVING COUNT(*) > 1
    ),
    multi_client AS (
        SELECT content_hash_id
        FROM month_mar
        GROUP BY 1
        HAVING COUNT(DISTINCT client_hash_id) > 1
    )
    SELECT 'duplicate day x client x content groups' AS check, COUNT(*) AS groups_found FROM duplicate_grain
    UNION ALL
    SELECT 'content items under more than one client', COUNT(*) FROM multi_client
    """
).df()

display(grain)
assert grain["groups_found"].sum() == 0, "Grain does not hold"
print("Grain holds: zero duplicate groups, and every page belongs to exactly one client.")

,check,groups_found
0,duplicate day x client x content groups,0
1,content items under more than one client,0


Grain holds: zero duplicate groups, and every page belongs to exactly one client.


### 3.2 Row count and date span — what does my slice actually cover?

Claim: `month=2026-03` is one complete calendar month of daily rows, with far more day-rows than
pages, and position is missing on a meaningful share of them. The availability-filtered subset of
these rows is the next check.

In [3]:
span = con.sql(
    """
    SELECT COUNT(*)                                        AS rows,
           COUNT(DISTINCT client_hash_id)                  AS clients,
           COUNT(DISTINCT content_hash_id)                 AS content_items,
           COUNT(DISTINCT report_date)                     AS days_covered,
           MIN(report_date)                                AS first_day,
           MAX(report_date)                                AS last_day,
           COUNT(*) FILTER (WHERE gsc_impressions > 0)     AS rows_with_impressions,
           COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) AS rows_without_position
    FROM month_mar
    """
).df()

display(span.T.rename(columns={0: "value"}))
assert pd.Timestamp(span.loc[0, "first_day"]).date().isoformat() == "2026-03-01"
assert pd.Timestamp(span.loc[0, "last_day"]).date().isoformat() == "2026-03-31"
print(f"March 2026 is complete: {int(span.loc[0, 'rows']):,} day-rows over "
      f"{int(span.loc[0, 'content_items']):,} pages and {int(span.loc[0, 'clients'])} clients.")

,value
rows,9841378
clients,55
content_items,331437
days_covered,31
first_day,2026-03-01 00:00:00
last_day,2026-03-31 00:00:00
rows_with_impressions,3611061
rows_without_position,6230317


March 2026 is complete: 9,841,378 day-rows over 331,437 pages and 55 clients.


### 3.3 Availability — which rows survive the `IS TRUE` filter?

Claim: the GSC availability flag is the honest gate, the GA4 flag is not usable at this scale, and
`IS TRUE` is the only safe comparison because the flags are three-valued (TRUE / FALSE / NULL).

In [4]:
avail = con.sql(
    """
    SELECT COUNT(*)                                                   AS fact_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)         AS gsc_is_true,
           COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE)     AS gsc_not_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)         AS ga4_is_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE)     AS ga4_not_true
    FROM month_mar
    """
).df()

rows = int(avail.loc[0, "fact_rows"])
summary = pd.DataFrame({
    "rows": [rows, int(avail.loc[0, "gsc_is_true"]), int(avail.loc[0, "ga4_is_true"])],
    "percent_of_fact_rows": [100.0,
                             100 * avail.loc[0, "gsc_is_true"] / rows,
                             100 * avail.loc[0, "ga4_is_true"] / rows],
}, index=["all March day-rows", "gsc_data_available IS TRUE", "ga4_data_available IS TRUE"])
display(summary.round(2))
assert avail.loc[0, "gsc_is_true"] > 0
print(f"The slice keeps {int(avail.loc[0, 'gsc_is_true']):,} of {rows:,} March day-rows "
      f"({100 * avail.loc[0, 'gsc_is_true'] / rows:.1f}%) under GSC IS TRUE. "
      f"Only {100 * avail.loc[0, 'ga4_is_true'] / rows:.1f}% are GA4 IS TRUE, so GA4 features would "
      "drop most of the panel; this contract is GSC-only.")

,rows,percent_of_fact_rows
all March day-rows,9841378,100.00
gsc_data_available IS TRUE,3611061,36.69
ga4_data_available IS TRUE,413966,4.21


The slice keeps 3,611,061 of 9,841,378 March day-rows (36.7%) under GSC IS TRUE. Only 4.2% are GA4 IS TRUE, so GA4 features would drop most of the panel; this contract is GSC-only.


### 3.4 Five features, and when each one is knowable

Built from the same month the checks above describe. Five, no more, and every one is complete on
the slice by construction — no imputation.

| Feature | Meaning | Available when? |
|---|---|---|
| `avg_position_mar` | Impression-weighted March position | Knowable on 2026-04-01 because March closed on 03-31; no April data is used. |
| `log_impressions_mar` | `log1p` of March impressions | Knowable on 2026-04-01 because a closed month's impression count is final. |
| `ctr_mar` | March clicks ÷ March impressions × 100 | Knowable on 2026-04-01 because both inputs are March measurements. |
| `days_with_impressions_mar` | March days with ≥ 1 impression (0–31) | Knowable on 2026-04-01 because it is a calendar fact about March. |
| `age_days_at_decision` | 2026-04-01 − `content_created_date` | Knowable on 2026-04-01 because the creation date is immutable metadata. |

This is a **feature build**, not a fourth check.

In [5]:
frame = con.sql(
    """
    WITH mar AS (
        SELECT content_hash_id,
               ANY_VALUE(client_hash_id) AS client_hash_id,
               SUM(gsc_impressions)      AS imp_mar,
               SUM(gsc_clicks)           AS clk_mar,
               SUM(CASE WHEN gsc_avg_position IS NOT NULL
                        THEN gsc_impressions * gsc_avg_position END)
                 / NULLIF(SUM(CASE WHEN gsc_avg_position IS NOT NULL
                                   THEN gsc_impressions END), 0) AS pos_mar,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_mar
        FROM month_mar
        WHERE gsc_data_available IS TRUE
        GROUP BY 1
    ),
    apr AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_apr,
               SUM(CASE WHEN gsc_avg_position IS NOT NULL
                        THEN gsc_impressions * gsc_avg_position END)
                 / NULLIF(SUM(CASE WHEN gsc_avg_position IS NOT NULL
                                   THEN gsc_impressions END), 0) AS pos_apr
        FROM month_apr
        WHERE gsc_data_available IS TRUE
        GROUP BY 1
    )
    SELECT m.client_hash_id, m.content_hash_id,
           m.imp_mar, m.clk_mar, m.pos_mar, m.days_with_impressions_mar, a.imp_apr, a.pos_apr,
           c.content_created_date, c.content_updated_date,
           DATE_DIFF('day', c.content_created_date, DATE '2026-04-01') AS age_days_at_decision
    FROM mar m
    JOIN apr a USING (content_hash_id)
    LEFT JOIN (SELECT content_hash_id, ANY_VALUE(content_created_date) AS content_created_date,
                      ANY_VALUE(content_updated_date) AS content_updated_date
               FROM dim_content GROUP BY 1) c USING (content_hash_id)
    WHERE m.imp_mar >= 100 AND a.imp_apr >= 100
      AND m.pos_mar IS NOT NULL AND a.pos_apr IS NOT NULL
    """
).df()

frame = frame.rename(columns={"pos_mar": "avg_position_mar", "pos_apr": "avg_position_apr"})
built = len(frame)
frame["log_impressions_mar"] = np.log1p(frame["imp_mar"])
frame["ctr_mar"] = 100 * frame["clk_mar"] / frame["imp_mar"]

FEATURES = ["avg_position_mar", "log_impressions_mar", "ctr_mar",
            "days_with_impressions_mar", "age_days_at_decision"]
LABEL = "avg_position_apr"

frame = frame.dropna(subset=FEATURES + [LABEL]).reset_index(drop=True)
assert len(FEATURES) == 5
assert frame["content_hash_id"].is_unique
assert frame[FEATURES].notna().all().all()

display(frame[FEATURES + [LABEL]].describe().T.round(2))
print(f"Feature frame: {len(frame):,} pages x {len(FEATURES)} features, "
      f"{frame['client_hash_id'].nunique()} clients, zero missing values "
      f"({built - len(frame):,} pages dropped for a missing feature).")

,count,mean,std,min,25%,50%,75%,max
avg_position_mar,88482.0,13.69,13.82,0.02,4.63,7.96,18.24,94.13
log_impressions_mar,88482.0,7.04,1.36,4.62,5.98,6.93,8.01,13.33
ctr_mar,88482.0,0.27,0.41,0.00,0.00,0.15,0.38,13.53
days_with_impressions_mar,88482.0,28.73,4.94,1.00,29.00,31.00,31.00,31.00
age_days_at_decision,88482.0,188.50,126.72,2.00,71.00,188.00,264.00,495.00
avg_position_apr,88482.0,15.20,13.91,0.07,5.58,9.47,20.80,92.26


Feature frame: 88,482 pages x 5 features, 42 clients, zero missing values (0 pages dropped for a missing feature).


### 3.5 The trap — add one label-derived column, watch the score, remove it

Same five features, same split. First an honest quick score, then **one** extra column pulled from
the month we are predicting (April's own position), then the same score again after deleting it.

In [6]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def quick_score(data, cols):
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    tr, va = next(splitter.split(data, groups=data["client_hash_id"]))
    train, valid = data.iloc[tr], data.iloc[va]
    assert set(train["client_hash_id"]).isdisjoint(set(valid["client_hash_id"]))

    baseline = float(train[LABEL].median())
    baseline_mae = mean_absolute_error(valid[LABEL], np.full(len(valid), baseline))
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(train[cols], train[LABEL])
    pred = model.predict(valid[cols])
    return {
        "baseline_mae": baseline_mae,
        "model_mae": mean_absolute_error(valid[LABEL], pred),
        "model_r2": r2_score(valid[LABEL], pred),
        "n_train": len(train),
        "n_valid": len(valid),
        "clients_valid": valid["client_hash_id"].nunique(),
    }


honest = quick_score(frame, FEATURES)
display(pd.DataFrame([honest]).T.rename(columns={0: "honest (5 features)"}).round(3))
print(f"Honest quick score: MAE {honest['model_mae']:.2f} position points "
      f"against a {honest['baseline_mae']:.2f} median baseline, R2 {honest['model_r2']:.2f}.")

,honest (5 features)
baseline_mae,7.229
model_mae,4.546
model_r2,0.597
n_train,82325.000
n_valid,6157.000
clients_valid,11.000


Honest quick score: MAE 4.55 position points against a 7.23 median baseline, R2 0.60.


In [7]:
# The deliberate mistake: pull the same metric from the month we are predicting.
leaky = frame.assign(gsc_avg_position_april=frame[LABEL])
leak = quick_score(leaky, FEATURES + ["gsc_avg_position_april"])
display(pd.DataFrame({"honest": honest, "with leaked column": leak}).round(3))
print(f"With the leaked column: MAE {leak['model_mae']:.3f}, R2 {leak['model_r2']:.3f} "
      "— the score jumps toward perfect.")

,honest,with leaked column
baseline_mae,7.229,7.229
model_mae,4.546,0.000
model_r2,0.597,1.000
n_train,82325.000,82325.000
n_valid,6157.000,6157.000
clients_valid,11.000,11.000


With the leaked column: MAE 0.000, R2 1.000 — the score jumps toward perfect.


In [8]:
leaky = leaky.drop(columns=["gsc_avg_position_april"])
assert "gsc_avg_position_april" not in leaky.columns

after = quick_score(leaky, FEATURES)
for key in ("baseline_mae", "model_mae", "model_r2"):
    assert np.isclose(after[key], honest[key]), f"honest score did not reproduce: {key}"
assert round(after["model_mae"], 6) != round(leak["model_mae"], 6)

display(pd.DataFrame({"honest": honest, "with leaked column": leak,
                      "after removal": after}).round(3))
print(f"Leak removed, honest number kept: MAE {after['model_mae']:.2f}, R2 {after['model_r2']:.2f}.")
print("The jump was not a better model. The column was the answer, measured after the decision moment.")

,honest,with leaked column,after removal
baseline_mae,7.229,7.229,7.229
model_mae,4.546,0.000,4.546
model_r2,0.597,1.000,0.597
n_train,82325.000,82325.000,82325.000
n_valid,6157.000,6157.000,6157.000
clients_valid,11.000,11.000,11.000


Leak removed, honest number kept: MAE 4.55, R2 0.60.
The jump was not a better model. The column was the answer, measured after the decision moment.


What the trap shows: the leaked column was not corrupted data — it was April position, the label
itself, wearing a normal column name. A column is leakage when it is measured **after** the
decision moment, even when it looks like an ordinary metric. The score returned exactly to the
honest value once it was removed, which is the only trustworthy number in this section.

## 4. Data limits

**Named limitation: my five features describe a page's search history, not its content.** The
warehouse delivers `dim_content` metadata (`word_count`, `content_type`) as a 2026-07-03 snapshot.
I cannot prove what those values were on 2026-04-01 — a page edited in May would carry its May word
count backwards into April. I excluded them rather than pretend otherwise. The consequence is
concrete: this slice can rank pages by momentum and visibility, but it cannot separate content
quality from search history, so a weak result here would not prove that content does not matter.

The cell below quantifies the snapshot problem from the frame already in memory (pandas only — this
is not a fourth warehouse query).

In [9]:
updated = frame["content_updated_date"]
after_decision = (updated > pd.Timestamp("2026-04-01")).mean() * 100
display(pd.DataFrame({
    "measure": ["pages in frame", "pages carrying an update date",
                "share of pages updated after 2026-04-01"],
    "value": [f"{len(frame):,}", f"{int(updated.notna().sum()):,}", f"{after_decision:.1f}%"],
}))
print("Metadata rows are demonstrably later than the decision moment, so word_count and "
      "content_type cannot be treated as April-1 values.")

,measure,value
0,pages in frame,"88,482"
1,pages carrying an update date,"88,482"
2,share of pages updated after 2026-04-01,82.1%


Metadata rows are demonstrably later than the decision moment, so word_count and content_type cannot be treated as April-1 values.


Two smaller limits, stated so they are not discovered later:

- The 100-impression floor in both months removes thin pages, so every conclusion is about pages
  that were already visible — not the long tail.
- This is an observational slice. It supports prioritisation, not the claim that editing a page
  moves its position.

## Self-check

- [x] Five contract answers written in plain words, before the query results were read
- [x] Exactly three verification queries, outputs visible, availability checked with `IS TRUE`
- [x] Five features, an "available when?" line for each, and no missing values in the frame
- [x] One label-derived column added, scored, deleted, and the honest number reproduced
- [x] One named limitation of the slice, with its consequence
- [x] Notebook runs top to bottom with no errors
- [x] No token, client name, URL, query text, or raw identifier appears in any output
- [x] Committed under `work/notebooks/`

Applied skills: [writing-data-contracts](../../skills/writing-data-contracts/SKILL.md) and
[flyrank-data](../../skills/flyrank/flyrank-data/SKILL.md). Column meanings follow
[docs/data-dictionary.md](../../docs/data-dictionary.md).